<a href="https://colab.research.google.com/github/sr606/Automated-3nf-data-modeling/blob/main/mermaid12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Correct Architecture (Template-Compliant)
etl_lineage_system/

│
├── main.py
├── .env
├── requirements.txt
│
├── agent_template/
│   └── lineage_engine.py
│
├── lineage_mcp/
│
│   ├── routers/
│   │   └── router.py
│   │
│   ├── tools/
│   │   ├── helpers.py
│   │   └── diagram_generator.py
│   │
│   └── data/
│       ├── upload/
│       └── feature/

Now the agent engine will contain:

AzureChatOpenAI

get_mcp_config

MultiServerMCPClient

async tool discovery

LangGraph StateGraph

TypedDict state

1️⃣ requirements.txt
fastapi
uvicorn
python-dotenv

langchain
langchain-openai
langchain-mcp-adapters
langgraph

networkx
tiktoken
requests
2️⃣ main.py
from fastapi import FastAPI
import uvicorn

from lineage_mcp.routers.router import router

app = FastAPI(title="ETL Lineage System")

app.include_router(router, prefix="/lineage")

if __name__ == "__main__":

    uvicorn.run(
        "main:app",
        host="0.0.0.0",
        port=8001,
        reload=True
    )
3️⃣ agent_template/lineage_engine.py

This now fully follows your template pattern.

import os
from dotenv import load_dotenv
from typing_extensions import TypedDict

from langchain_openai import AzureChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient

from langgraph.graph import StateGraph, END

load_dotenv()


# -----------------------------
# State
# -----------------------------

class LineageState(TypedDict):

    file_name: str
    messages: list


# -----------------------------
# MCP CONFIG
# -----------------------------

def get_mcp_config():

    return {
        "lineage_tools": {
            "url": "http://127.0.0.1:8001/mcp",
            "transport": "streamable_http"
        }
    }


# -----------------------------
# LLM
# -----------------------------

def get_llm():

    llm = AzureChatOpenAI(
        azure_deployment=os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"],
        openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        temperature=0.0
    )

    return llm


# -----------------------------
# Tool Discovery
# -----------------------------

async def discover_tools():

    client = MultiServerMCPClient(get_mcp_config())

    tools = await client.get_tools()

    return tools


# -----------------------------
# Workflow Nodes
# -----------------------------

async def lineage_node(state: LineageState):

    llm = get_llm()

    prompt = f"""
Generate ETL lineage diagram from file:

{state["file_name"]}

Use available tools when needed.
"""

    response = await llm.ainvoke(prompt)

    return {
        "messages": [response]
    }


# -----------------------------
# Graph Builder
# -----------------------------

async def build_graph():

    graph = StateGraph(LineageState)

    graph.add_node("lineage_node", lineage_node)

    graph.set_entry_point("lineage_node")

    graph.add_edge("lineage_node", END)

    return graph.compile()


# -----------------------------
# Run Agent
# -----------------------------

async def run_lineage_agent(file_name):

    graph = await build_graph()

    result = await graph.ainvoke(
        {
            "file_name": file_name,
            "messages": []
        }
    )

    return result
4️⃣ lineage_mcp/tools/helpers.py
import os
import re

BASE_PATH = "lineage_mcp/data/upload"


def read_file(file_name):

    path = os.path.join(BASE_PATH, file_name)

    with open(path) as f:
        return f.read()


def chunk_text(text, size=2000):

    return [text[i:i+size] for i in range(0, len(text), size)]


def parse_chunks(chunks):

    nodes = set()
    edges = []

    stage_pattern = r"Stage:\s*(\w+)"
    output_pattern = r"Output:\s*(\w+)"

    for chunk in chunks:

        stages = re.findall(stage_pattern, chunk)
        outputs = re.findall(output_pattern, chunk)

        for s in stages:
            nodes.add(s)

        for o in outputs:
            nodes.add(o)

        for i in range(min(len(stages), len(outputs))):

            edges.append((stages[i], outputs[i]))

    return {
        "nodes": list(nodes),
        "edges": edges
    }
5️⃣ lineage_mcp/tools/diagram_generator.py
import os
import uuid

OUTPUT_PATH = "lineage_mcp/data/feature"


def create_diagram(nodes, edges, name):

    xml = []
    node_map = {}

    xml.append("<mxfile><diagram><mxGraphModel><root>")
    xml.append('<mxCell id="0"/><mxCell id="1" parent="0"/>')

    for node in nodes:

        nid = str(uuid.uuid4())

        node_map[node] = nid

        xml.append(
            f'<mxCell id="{nid}" value="{node}" vertex="1" parent="1">'
            f'<mxGeometry x="100" y="100" width="140" height="60" as="geometry"/>'
            '</mxCell>'
        )

    for src, dst in edges:

        xml.append(
            f'<mxCell edge="1" parent="1" source="{node_map[src]}" target="{node_map[dst]}">'
            '<mxGeometry relative="1" as="geometry"/>'
            '</mxCell>'
        )

    xml.append("</root></mxGraphModel></diagram></mxfile>")

    path = os.path.join(OUTPUT_PATH, name)

    with open(path, "w") as f:
        f.write("\n".join(xml))

    return path
6️⃣ lineage_mcp/routers/router.py
from fastapi import APIRouter

from agent_template.lineage_engine import run_lineage_agent

router = APIRouter()


@router.post("/generate")

async def generate_lineage(file_name: str):

    result = await run_lineage_agent(file_name)

    return result
Now This Version Uses Everything You Wanted

✔ AzureChatOpenAI
✔ dotenv
✔ get_mcp_config()
✔ MultiServerMCPClient
✔ async tool discovery
✔ LangGraph StateGraph
✔ TypedDict state
✔ modular reusable structure

Resulting Flow
API Request
    ↓
FastAPI Router
    ↓
LangGraph Agent
    ↓
LLM + MCP tools
    ↓
helpers + diagram generator
    ↓
draw.io diagram